# Entrenamiento de YOLOv8 para detección de armas visibles

Cuaderno del objetivo específico 4. Se ejecuta en **Google Colab con GPU gratuita**.

Antes de empezar: `Entorno de ejecución` → `Cambiar tipo de entorno` → `T4 GPU`.

El resultado es un archivo `best.pt` de unos 6 MB que cabe sin problema en el
repositorio y que la estación de captura carga con `DetectorArmas`.

---

## Qué dice el documento y qué hay que medir

El trabajo afirma que las armas blancas son más difíciles de detectar que las
de fuego. Esa afirmación hay que sostenerla con números, así que la evaluación
reporta las métricas **separadas por clase**, no un promedio global que esconde
la diferencia.

> **Si ve un `UnpicklingError` al entrenar:** es un choque de versiones
> entre Ultralytics y PyTorch, no un problema de su dataset. Asegurese de
> que la celda de instalacion diga `ultralytics>=8.3.0`, luego
> `Entorno de ejecucion` -> `Reiniciar sesion`, y vuelva a ejecutar desde
> el principio.


## 1. Comprobar la GPU

Si esta celda no muestra una tarjeta, el entrenamiento tardará horas en vez de
minutos. Cambie el entorno de ejecución antes de seguir.

In [ ]:
!nvidia-smi

In [ ]:
# Ultralytics 8.2 no es compatible con el PyTorch que trae Colab hoy:
# desde PyTorch 2.6 torch.load usa weights_only=True por defecto y la
# version vieja falla al cargar los pesos con UnpicklingError.
!pip install -q "ultralytics>=8.3.0" roboflow

import ultralytics
ultralytics.checks()
print('\nVersion de ultralytics:', ultralytics.__version__)
print('Anote esta version en el documento, para que el entrenamiento\n'
      'sea reproducible.')


## 2. Descargar el dataset

En Roboflow Universe busque un dataset de *weapon detection* que tenga
**pistolas y cuchillos etiquetados por separado**. Entre en el dataset,
`Download this Dataset`, formato **YOLOv8**, `show download code`, y pegue
aquí las tres líneas que le entrega.

Qué mirar antes de elegir uno:

- Que tenga **las dos clases**. Muchos datasets solo traen armas de fuego, y
  con ese no puede sostener la comparación que promete el documento.
- Que las imágenes se parezcan a una portería: personas de cuerpo entero, a
  varios metros, con luz de interior. Un dataset de primeros planos sobre fondo
  blanco da métricas altísimas que después no se reproducen.
- Al menos unas 1.500 imágenes de entrenamiento.

In [ ]:
from roboflow import Roboflow

# Reemplace con el código que le da Roboflow
rf = Roboflow(api_key="SU_API_KEY")
proyecto = rf.workspace("WORKSPACE").project("PROYECTO")
dataset = proyecto.version(1).download("yolov8")

print("Dataset en:", dataset.location)

## 3. Revisar el dataset antes de entrenar

Media hora mirando los datos ahorra un entrenamiento perdido. Esta celda
cuenta cuántas instancias hay de cada clase: si una está muy por debajo de la
otra, el modelo la detectará peor y conviene saberlo **antes**, no después.

In [ ]:
import collections
import pathlib

import yaml

raiz = pathlib.Path(dataset.location)
config = yaml.safe_load((raiz / "data.yaml").read_text())
nombres = config["names"]
print("Clases del dataset:", nombres)

for particion in ("train", "valid", "test"):
    carpeta = raiz / particion / "labels"
    if not carpeta.exists():
        continue
    conteo = collections.Counter()
    imagenes = 0
    for etiqueta in carpeta.glob("*.txt"):
        imagenes += 1
        for linea in etiqueta.read_text().splitlines():
            if linea.strip():
                conteo[nombres[int(linea.split()[0])]] += 1
    print(f"\n{particion}: {imagenes} imágenes")
    for clase, total in conteo.most_common():
        print(f"   {clase:20s} {total:6d} instancias")

## 4. Entrenar

`yolov8n` es la variante más pequeña. Se elige a propósito: la estación de
captura es un portátil sin GPU, y el RNF-02 exige menos de dos segundos por
fotograma. Un modelo más grande daría mejores métricas en el papel y no
cumpliría el requisito en el equipo real.

Con unas 1.500 imágenes y T4, 50 épocas toman entre 40 y 90 minutos.

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolov8n.pt")          # pesos preentrenados en COCO

resultados = modelo.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,                      # corta si deja de mejorar
    project="faceguard",
    name="armas_v1",
    seed=42,                          # reproducibilidad para el documento
    # Aumentos pensados para una portería real
    hsv_v=0.5,                        # variación de brillo: luz de interior
    degrees=10,                       # la cámara nunca está perfectamente recta
    scale=0.5,                        # personas a distintas distancias
    fliplr=0.5,
    mosaic=1.0,
)

## 5. Métricas separadas por clase

Esto es lo que va al Capítulo IV. El promedio global no sirve: hay que mostrar
pistolas y cuchillos por separado.

In [ ]:
mejor = YOLO("faceguard/armas_v1/weights/best.pt")
metricas = mejor.val(data=f"{dataset.location}/data.yaml", split="test")

print(f"\n{'Clase':<22}{'Precisión':>11}{'Exhaustiv.':>12}{'F1':>8}{'mAP50':>9}")
print("-" * 62)
for indice, clase in enumerate(metricas.names.values()):
    p, r, ap50, _ = metricas.box.class_result(indice)
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    print(f"{clase:<22}{p:>11.3f}{r:>12.3f}{f1:>8.3f}{ap50:>9.3f}")

print("-" * 62)
print(f"{'Global (mAP50)':<22}{'':>11}{'':>12}{'':>8}{metricas.box.map50:>9.3f}")

## 6. Tiempo de inferencia en CPU

El RNF-02 pide menos de dos segundos por fotograma, y la estación de captura
**no tiene GPU**. Medir en la T4 de Colab no demuestra nada, así que esta celda
fuerza CPU para que el número se parezca al del equipo real.

In [ ]:
import time

import numpy as np

en_cpu = YOLO("faceguard/armas_v1/weights/best.pt")
fotograma = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)

en_cpu.predict(fotograma, device="cpu", verbose=False)   # descarta la primera

tiempos = []
for _ in range(20):
    inicio = time.perf_counter()
    en_cpu.predict(fotograma, device="cpu", verbose=False)
    tiempos.append((time.perf_counter() - inicio) * 1000)

print(f"Mediana: {np.median(tiempos):.0f} ms")
print(f"Máximo:  {np.max(tiempos):.0f} ms")
print(f"RNF-02 (menos de 2000 ms): "
      f"{'CUMPLE' if np.max(tiempos) < 2000 else 'NO CUMPLE'}")

## 7. Casos negativos

La asesoría pide expresamente que un celular, una sombrilla o una mano en
forma de pistola **no** disparen la alerta. Suba aquí unas 10 fotos de esos
objetos: cada detección que aparezca es un falso positivo, y el falso positivo
es el riesgo real del módulo, porque un guardia que recibe avisos equivocados
termina ignorándolos todos.

In [ ]:
from google.colab import files

subidas = files.upload()

UMBRAL = 0.70        # el mismo de Config.UMBRAL_ARMA
falsos = 0

for nombre in subidas:
    salida = mejor.predict(nombre, conf=UMBRAL, verbose=False)[0]
    detecciones = [(salida.names[int(c.cls)], float(c.conf)) for c in salida.boxes]
    if detecciones:
        falsos += 1
        print(f"  FALSO POSITIVO  {nombre}: {detecciones}")
    else:
        print(f"  correcto        {nombre}: sin alerta")

print(f"\nFalsos positivos: {falsos} de {len(subidas)}")
if falsos:
    print("Suba el umbral en UMBRAL_ARMA o añada esas imágenes como "
          "negativos al dataset y reentrene.")

## 8. Descargar los pesos

Guarde `best.pt` como `modelos/yolov8n-armas.pt` en el repositorio y apunte
`MODELO_YOLO` a esa ruta en el archivo `.env` de la estación de captura.

Descargue también la matriz de confusión y las curvas: son las figuras del
Capítulo IV.

In [ ]:
from google.colab import files

files.download("faceguard/armas_v1/weights/best.pt")

# Figuras para el documento
for figura in ("confusion_matrix.png", "results.png",
               "PR_curve.png", "F1_curve.png"):
    ruta = f"faceguard/armas_v1/{figura}"
    try:
        files.download(ruta)
    except Exception:
        print(f"(no se generó {figura})")

---

## Si las métricas salen bajas

Es lo normal en el primer intento y no es un fracaso, es un resultado que el
documento puede reportar con honestidad.

| Qué ve | Qué significa | Qué hacer |
|---|---|---|
| Exhaustividad baja en cuchillos | Se le escapan armas blancas | Más imágenes de esa clase; es el hallazgo que el documento ya anticipa |
| Precisión baja en general | Alarma de más | Suba `UMBRAL_ARMA`; añada negativos al dataset |
| Buenas métricas en test y malas en la prueba real | El dataset no se parece a una portería | Añada fotos tomadas en el punto de acceso real |
| Nada mejora tras la época 20 | Convergió | `patience` ya lo corta; no insista con más épocas |

Un modelo con 0.70 de F1 en cuchillos y 0.90 en pistolas, reportado y
explicado, vale más que un 0.95 global que no se sostiene en la demostración.